In [1]:
# Load the data
import pandas as pd

DATA = "../data/raw/ml-32m" # "../data/raw/ml-latest-small"

ratings = pd.read_csv(DATA + "/ratings.csv")
movies = pd.read_csv(DATA + "/movies.csv")

number_of_users = ratings['userId'].max()

In [2]:
MIN_USER_RATINGS = 5
MIN_MOVIE_RATINGS = 20

In [3]:
# Drop movies with fewer than MIN_MOVIE_RATINGS ratings
movie_counts = ratings.groupby("movieId")["rating"].transform("size")
ratings = ratings[movie_counts >= MIN_MOVIE_RATINGS]

# Drop users with fewer than MIN_USER_RATINGS ratings
user_counts = ratings.groupby("userId")["rating"].transform("size")
ratings = ratings[user_counts >= MIN_USER_RATINGS]

print(len(ratings), ratings["movieId"].nunique(), ratings["userId"].nunique())

31725920 23350 200948


In [4]:
# Remap IDs to contiguous indices 0..N-1
ratings["movie_idx"], movie_ids = pd.factorize(ratings["movieId"], sort=True)
ratings["user_idx"], user_ids = pd.factorize(ratings["userId"], sort=True)

print(ratings["movie_idx"].max(), ratings["user_idx"].max())

23349 200947


In [5]:
# Lookup table: movie_idx -> movieId -> title
movie_table = pd.DataFrame({"movie_idx": range(len(movie_ids)), "movieId": movie_ids})
movie_table = movie_table.merge(movies, on="movieId", how="left")

print(len(movie_table), movie_table["title"].isna().sum())
movie_table.head()

23350 0


,movie_idx,movieId,title,genres
0,0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,2,Jumanji (1995),Adventure|Children|Fantasy
2,2,3,Grumpier Old Men (1995),Comedy|Romance
3,3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,4,5,Father of the Bride Part II (1995),Comedy


In [6]:
# Sort each user's ratings oldest to newest
ratings = ratings.sort_values(["user_idx", "timestamp", "movie_idx"])

# Count backward from each user's newest rating: 0 = latest, 1 = second latest, ...
from_end = ratings.groupby("user_idx").cumcount(ascending=False)

train = ratings[from_end >= 2]
val = ratings[from_end == 1]
test = ratings[from_end == 0]

seen = set(train["movie_idx"])
val = val[val["movie_idx"].isin(seen)]
test = test[test["movie_idx"].isin(seen)]

print(len(train), len(val), len(test))

31324024 200948 200948


In [7]:
OUT = "../data/processed/ml-32m" # "../data/processed/ml-latest-small"

cols = ["user_idx", "movie_idx", "rating", "timestamp"]
train[cols].to_csv(f"{OUT}/train.csv", index=False)
val[cols].to_csv(f"{OUT}/val.csv", index=False)
test[cols].to_csv(f"{OUT}/test.csv", index=False)
movie_table.to_csv(f"{OUT}/movie_table.csv", index=False)

print(len(user_ids), len(movie_ids), train["rating"].mean())

200948 23350 3.5429159420896883
